In [54]:
%pip install -U langchain langchain-core "langchain-community<=0.4" langchain-anthropic rapidfuzz langchain-experimental langchain-text-splitters langchain-neo4j langchain-huggingface langchain-ollama neo4j python-dotenv sentence-transformers ragas

  Using cached langchain_anthropic-1.4.8-py3-none-any.whl.metadata (3.5 kB)
  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached anthropic-0.116.0-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.
Using cached langchain_anthropic-1.4.8-py3-none-any.whl (52 kB)
Using cached anthropic-0.116.0-py3-none-any.whl (956 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-anthropic]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from dotenv import load_dotenv
import os
import json

from copy import deepcopy


from pydantic import BaseModel, Field
from neo4j import GraphDatabase, Driver

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

from langchain_experimental.graph_transformers import LLMGraphTransformer

from langchain_community.graphs.graph_document import GraphDocument
from langchain_core.documents import Document
from langchain_community.graphs.graph_document import Node, Relationship

from typing import List, Dict, Any

from pydantic import BaseModel, Field, ValidationError
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs.graph_document import Node, Relationship, GraphDocument

from ragas import EvaluationDataset

from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

load_dotenv()

d:\dev\holmes-comparison-grag-rag\local-13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [7]:
loader = TextLoader(file_path="../data/chapters/chapter_1.txt", encoding="utf-8")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
documents = text_splitter.split_documents(documents=docs)

with open("../data/chunks/chunked_chapter_1.txt", "w", encoding="utf-8") as file_out:
    file_out.write(str(documents))

In [8]:
documents[0].page_content

'I. A SCANDAL IN BOHEMIA\n\n\nI.\n\nTo Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe world has seen, but as a lover he would have placed himself in a\nfalse position. He never spoke of the softer passions, save with a gibe\nand a sneer. They were admirable things for the observer—excellent for\ndrawing the veil from men’s motives and actions. But for the trained\nreasoner to admit such intrusions into his own delicate and finely\nadjusted temperament was to introduce a distracting factor which might\nthrow a doubt upon all his mental results. Grit in a sensitive\ninstrument, or a crack in one of his own high-power lens

In [87]:
graph = Neo4jGraph(database="local2")

In [8]:
def serialize_node(node):
    return Node(id=node["id"], type=node["type"], properties=node["properties"])


def serialize_relationship(relation, nodes: list[Node]):
    source = Node(id=relation["source"]["id"], type=relation["source"]["type"], properties=relation["source"]["properties"])
    target = Node(id=relation["target"]["id"], type=relation["target"]["type"], properties=relation["target"]["properties"])
    return Relationship(
        source=source,
        target=target,
        type=relation["type"],
        properties=relation["properties"],
    )

def cast_to_graph_document(json_object:any):
    nodes: list[Node] = []
    for node in json_object["nodes"]:
        nodes.append(serialize_node(node))

    relationships: list[Relationship] = []
    for rel in json_object["relationships"]:
        relationships.append(serialize_relationship(rel, nodes))
    
    source: Document = Document(metadata=json_object["document"]["metadata"], page_content=json_object["document"]["page_content"])
    return GraphDocument(nodes=nodes, relationships=relationships, source=source)


def convert_to_graph_doc(s: str):
    graph_documents = eval(s, {
        "GraphDocument": GraphDocument,
        "Document": Document,
        "Node": Node,
        "Relationship": Relationship,
    })
    return graph_documents

In [9]:
class NodeModel(BaseModel):
    id: str
    type: str
    properties: Dict[str, Any] = Field(default_factory=dict)


class RelationshipModel(BaseModel):
    source: NodeModel
    target: NodeModel
    type: str
    properties: Dict[str, Any] = Field(default_factory=dict)


class DocumentModel(BaseModel):
    metadata: Dict[str, Any] = Field(default_factory=dict)
    page_content: str


class ExtractionResult(BaseModel):
    nodes: List[Node] = Field(default_factory=list)
    relationships: List[Relationship] = Field(default_factory=list)
    document: Document

In [10]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a high-recall literary information extraction system for building a knowledge graph from a single source document.

Your task:
Extract all important entities, attributes, and relationships explicitly stated in the input document, and return exactly one valid JSON object with the keys:
- nodes
- relationships
- document

Primary objective:
Maximize useful recall for downstream question answering over a literary corpus.
Prefer extracting too many relevant explicit facts over omitting important ones.

Hard rules:
- Output only valid JSON.
- Do not output markdown, code fences, commentary, or explanations.
- Do not output Python objects such as GraphDocument(...).
- Use only facts explicitly stated in the input document.
- Do not invent facts.
- Do not use background knowledge from outside the document.
- Process the entire input document as one unit.
- If no entities or relationships are present, return empty lists.

Strict schema:
- Every node.type must be exactly one of:
  Person, Organization, Place, Document, Artifact, Event, Case, Role
- Every relationship.type must be exactly one of:
  HAS_ROLE, ALIAS_OF, AFFILIATED_WITH, LOCATED_IN, RESIDES_AT, PARTICIPATED_IN, OCCURRED_AT, OCCURRED_ON, CREATED, ADDRESSED_TO, POSSESSES, TARGETS, RELATED_TO_CASE
- Do not use any other node types or relationship types.

Critical property constraint:
- The properties field of every node and relationship must be a flat JSON object.
- Allowed property values are only:
  - string
  - number
  - boolean
  - list of strings
  - list of numbers
  - list of booleans
- Do NOT use nested objects, nested dictionaries, maps, or lists of objects inside properties.
- If a detail would naturally be structured as an object, flatten it into one or more primitive properties instead.
- Example: use "hair_color": "black" and "face_description": "pale, refined" instead of nested appearance objects.
- Example: use "surface_forms": ["Holmes", "Mr. Holmes"] instead of nested alias objects.

What to extract:
Extract all explicit, question-relevant information, including:

1. Persons
- named characters
- partially named characters if no fuller name appears
- unnamed but clearly important referential persons, e.g. "the landlady", "the old man", "the bride", "the doctor"

2. Places
- addresses, rooms, buildings, streets, cities, regions, countries
- residences and places visited

3. Organizations and groups
- families, houses, institutions, police, royal houses, clubs, employers, professions where relevant

4. Important objects and documents
- letters, notes, photographs, weapons, tools, clothing, disguises, jewelry, keys, furniture, animals, vehicles, etc.
- include plot-relevant objects even if not named formally

5. Events and actions
- meetings, departures, marriages, threats, discoveries, thefts, disguises, conversations, journeys, attacks, investigations, requests, observations
- create event nodes when the event itself is important for later reasoning or connects multiple entities

6. Descriptive attributes
Store important explicit attributes in node.properties, especially:
- physical appearance: age, approximate age, hair, beard, eyes, face, complexion, scars, build, height, clothing, posture, expression
- identity and naming: titles, aliases, alternate names, epithets, descriptions
- social and personal attributes: occupation, profession, rank, family role, marital status, nationality, social status
- emotional or mental state: frightened, agitated, calm, angry, drunk, ill, tired, etc.
- location or residence
- possessions or associated objects
- distinguishing features useful for retrieval or disambiguation
- any other explicit traits likely to matter for question answering

Coreference and canonicalization:
- Resolve pronouns and shortened mentions to the most complete explicit name that appears in the same document.
- Examples:
  - "Holmes", "Mr. Holmes", "he" -> "Sherlock Holmes" if "Sherlock Holmes" appears in the document
  - "Watson", "I", "the narrator" -> "Dr. Watson" if "Dr. Watson" appears in the document
  - "the woman" -> "Irene Adler" only if "Irene Adler" explicitly appears in the same document and the reference is clearly unambiguous
- If the fuller canonical name does not appear in the document, keep the explicit local mention as its own node.
- Do not merge uncertain identities.
- Prefer preserving a distinct explicit entity over making a wrong merge.

Allowed node schemas:
- Person:
  - id: canonical human-readable name
  - type: "Person"
  - properties may include: canonical_name, surface_forms, gender, nationality, description, age, occupation, role, title, marital_status, residence, appearance, hair_color, beard, eye_description, face_description, complexion, build, height_description, scar_description, posture, expression, emotional_state, aliases
- Organization:
  - properties may include: name, org_type, surface_forms
- Place:
  - properties may include: name, place_type, surface_forms
- Document:
  - properties may include: title_or_label, doc_type, quoted_text, date_text, language
- Artifact:
  - properties may include: name, artifact_type, description
- Event:
  - properties may include: event_type, summary, date_text, sequence, certainty
- Case:
  - properties may include: case_name, summary, status
- Role:
  - properties may include: role_name, role_type, temporal_scope

Node requirements:
- Each node must have:
  - id: canonical human-readable identifier, preferably the most complete explicit form in the document
  - type: exactly one of the allowed node types above
  - properties: flat JSON object with primitive values only

Relationship requirements:
- Extract all important explicit relationships between nodes.
- Relationship types are restricted to the allowed list above.
- Each relationship must have:
  - source: node object
  - target: node object
  - type: exactly one of the allowed relationship types
  - properties: flat JSON object with primitive values only

Event extraction guidance:
- Create event nodes only when useful, e.g. a marriage, departure, attack, discovery, consultation, or theft.
- Link participants to events with relations such as PARTICIPATED_IN, OCCURRED_AT, OCCURRED_ON, CREATED, ADDRESSED_TO, POSSESSES, TARGETS, RELATED_TO_CASE.
- Do not create trivial event nodes for every sentence.

Deduplication:
- Each real-world entity should appear only once per document after clear coreference resolution.
- Do not create duplicate nodes that differ only by shortened mentions when the fuller canonical mention is explicit and unambiguous in the same document.

Document object:
- Return the input document unchanged under the key "document".
- Preserve its metadata and page_content.

Output format:
Return exactly one JSON object with exactly these top-level keys:
- nodes
- relationships
- document
"""
    ),
    (
        "human",
        "Input document:\n{input_document}"
    )
])


llm = ChatOllama(
    model="qwen3.6:latest",
    base_url="http://192.168.178.67:11434",
    temperature=0,
    # format=ExtractionResult.model_json_schema(),
    reasoning=False,
    keep_alive="24h",
    num_predict=-1,
    format="json"
)

llm_json = llm.with_structured_output(ExtractionResult, include_raw=True)


json_chain = prompt | llm_json

# response = json_chain.invoke({"input_document":json.dumps(
#                 {
#                     "metadata": documents[2].metadata,
#                     "page_content": documents[2].page_content,
#                 },
#                 ensure_ascii=False,
#             )})

### AI

In [11]:
from copy import deepcopy
from typing import Dict, Tuple
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship

def _normalize_id(node_id: str, alias_map: Dict[str, str]) -> str:
    key = node_id.strip().lower()
    return alias_map.get(key, node_id.strip())

def _merge_properties(a: dict, b: dict) -> dict:
    merged = deepcopy(a) if a else {}
    for k, v in (b or {}).items():
        if k not in merged:
            merged[k] = v
        else:
            if merged[k] == v:
                continue
            if isinstance(merged[k], list):
                existing = merged[k]
            else:
                existing = [merged[k]]
            if isinstance(v, list):
                for item in v:
                    if item not in existing:
                        existing.append(item)
            else:
                if v not in existing:
                    existing.append(v)
            merged[k] = existing
    return merged

def _canonical_node(node: Node, alias_map: Dict[str, str]) -> Node:
    new_node = deepcopy(node)
    new_node.id = _normalize_id(str(node.id), alias_map)
    return new_node

def resolve_results(validated_result: GraphDocument) -> GraphDocument:
    alias_map = {
        "holmes": "Sherlock Holmes",
        "sherlock": "Sherlock Holmes",
        "mr. holmes": "Sherlock Holmes",
        "watson": "Dr. John H. Watson",
        "dr watson": "Dr. John H. Watson",
        "the narrator": "Dr. John H. Watson",
        "narrator": "Dr. John H. Watson",
        "the woman": "Irene Adler",
        "miss adler": "Irene Adler",
        "irene norton": "Irene Adler",
        "i": "Dr. John H. Watson",
        "he": "Sherlock Holmes"
    }

    canonical_nodes: Dict[Tuple[str, str], Node] = {}

    for node in validated_result.nodes:
        cnode = _canonical_node(node, alias_map)
        key = (str(cnode.id), str(cnode.type))
        if key not in canonical_nodes:
            canonical_nodes[key] = cnode
        else:
            canonical_nodes[key].properties = _merge_properties(
                canonical_nodes[key].properties,
                cnode.properties
            )

    resolved_relationships = []

    for rel in validated_result.relationships:
        new_rel = deepcopy(rel)
        new_rel.source = _canonical_node(rel.source, alias_map)
        new_rel.target = _canonical_node(rel.target, alias_map)

        source_key = (str(new_rel.source.id), str(new_rel.source.type))
        target_key = (str(new_rel.target.id), str(new_rel.target.type))

        if source_key in canonical_nodes:
            new_rel.source = canonical_nodes[source_key]
        else:
            canonical_nodes[source_key] = new_rel.source

        if target_key in canonical_nodes:
            new_rel.target = canonical_nodes[target_key]
        else:
            canonical_nodes[target_key] = new_rel.target

        resolved_relationships.append(new_rel)

    deduped_relationships = []
    seen_rels = set()

    for rel in resolved_relationships:
        rel_key = (
            str(rel.source.id),
            str(rel.source.type),
            str(rel.type),
            str(rel.target.id),
            str(rel.target.type),
            json.dumps(rel.properties, sort_keys=True, ensure_ascii=False),
        )
        if rel_key not in seen_rels:
            seen_rels.add(rel_key)
            deduped_relationships.append(rel)

    return GraphDocument(
        nodes=list(canonical_nodes.values()),
        relationships=deduped_relationships,
        source=validated_result.source
    )

In [13]:
def extract_graph_document(resolved: ExtractionResult):
    return GraphDocument(nodes=resolved.nodes, relationships=resolved.relationships, source=resolved.document)

# def resolve_results(validated_result: GraphDocument):
#     resolved_nodes = []
#     resolved_relationships = []
#     alias_map = {
#         "holmes": "Sherlock Holmes",
#         "sherlock": "Sherlock Holmes",
#         "watson": "Dr. Watson",
#         "the woman": "Irene Adler",
#         "miss adler": "Irene Adler",
#         "mr. holmes": "Sherlock Holmes",
#         "i": "Dr. Watson",
#         "narrator": "Dr. Watson",
#         "he": "Sherlock Holmes"
#     }
#     for node in validated_result.nodes:
#         new_node = deepcopy(node)
#         if node.id.lower() in alias_map:
#             new_node.id = alias_map[node.id.lower()]
#             resolved_nodes.append(new_node)

#     for rel in validated_result.relationships:
#         source_node = rel.source
#         target_node = rel.target
#         new_rel = deepcopy(rel)
#         if source_node.id.lower() in alias_map:
#             new_node = deepcopy(source_node)
#             new_node.id = alias_map[source_node.id.lower()]
#             new_rel.source = new_node
#         if target_node.id.lower() in alias_map:
#             new_node = deepcopy(target_node)
#             new_node.id = alias_map[target_node.id.lower()]
#             new_rel.target = new_node
        
#         resolved_relationships.append(new_rel)
    
#     return GraphDocument(nodes=resolved_nodes, relationships=resolved_relationships, source=validated_result.source)


def extract_graph_document_json_enforced(doc: Document) -> GraphDocument:
    response = json_chain.invoke({"input_document": doc})
    # print(response)
    # json_data = json.loads(response.content)
    # print(f"json: {json_data}")

    validated = ExtractionResult.model_validate(response["parsed"])
    graph_doc = extract_graph_document(validated)
    return resolve_results(graph_doc)


def process_documents_json_enforced(documents: List[Document]) -> List[GraphDocument]:
    graph_docs = []
    for i, doc in enumerate(documents):
        try:
            graph_doc = extract_graph_document_json_enforced(doc)
            graph_docs.append(graph_doc)
            print(f"Processed document {i+1}/{len(documents)}")
        except (ValidationError, json.JSONDecodeError, Exception) as e:
            print(f"Failed document {i+1}: {e}")
    return graph_docs



In [22]:
response = json_chain.invoke(
        {
            "input_document": documents[1]
        }
    )
response

{'raw': AIMessage(content='{\n  "document": {\n    "page_content": "I had seen little of Holmes lately. My marriage had drifted us away\\nfrom each other. My own complete happiness, and the home-centred\\ninterests which rise up around the man who first finds himself master\\nof his own establishment, were sufficient to absorb all my attention,\\nwhile Holmes, who loathed every form of society with his whole Bohemian\\nsoul, remained in our lodgings in Baker Street, buried among his old\\nbooks, and alternating from week to week between cocaine and ambition, the drowsiness of the drug, and the fierce energy of his own keen\\nnature. He was still, as ever, deeply attracted by the study of crime,\\nand occupied his immense faculties and extraordinary powers of\\nobservation in following out those clues, and clearing up those\\nmysteries which had been abandoned as hopeless by the official police.\\nFrom time to time I heard some vague account of his doings: of his summons to Odessa in th

In [9]:
graph_docs = process_documents_json_enforced(documents[:1])


KeyboardInterrupt: 

In [15]:
# graph_docs = []
# for doc in documents[:5]:

# directory = "../data/chapters/"
# for file in os.listdir(directory):
#     print(f"processing {file}...")
loader = TextLoader(file_path=f"../data/chapters/chapter_1.txt", encoding="utf-8")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
documents = text_splitter.split_documents(documents=docs)

graph_docs = process_documents_json_enforced(documents[:15])

with open(f"../data/local_2/chapter_1_15.txt", "w") as file_out:
    file_out.write(str(graph_docs))

Processed document 1/15
Processed document 2/15
Processed document 3/15
Processed document 4/15
Processed document 5/15
Processed document 6/15
Processed document 7/15
Processed document 8/15
Processed document 9/15
Processed document 10/15
Processed document 11/15
Processed document 12/15
Processed document 13/15
Processed document 14/15
Processed document 15/15


In [ ]:
# clean_graph()
# graph.add_graph_documents(
#     parsed_graph_document,
#     baseEntityLabel=True,
#     include_source=True,
# )

In [7]:
import os
import json
from collections.abc import Mapping

def flatten_dict(d, parent_key="", sep="_"):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, Mapping):
            items.update(flatten_dict(v, new_key, sep=sep))
        elif isinstance(v, list):
            cleaned = []
            for item in v:
                if isinstance(item, Mapping):
                    cleaned.append(json.dumps(item, ensure_ascii=False))
                else:
                    cleaned.append(item)
            items[new_key] = cleaned
        else:
            items[new_key] = v
    return items

def sanitize_graph_document(graph_doc):
    for node in graph_doc.nodes:
        if node.properties:
            node.properties = flatten_dict(node.properties)
    for rel in graph_doc.relationships:
        if rel.properties:
            rel.properties = flatten_dict(rel.properties)
    return graph_doc

directory = "../data/local/"

# for filename in os.listdir(directory):
with open(os.path.join(directory, "chapter_1.txt"), "r", encoding="utf-8") as file_in:
    string_graph_docs = file_in.read()

parsed_graph_document = convert_to_graph_doc(string_graph_docs)

sanitized_docs = [sanitize_graph_document(doc) for doc in parsed_graph_document]

    # graph.add_graph_documents(
    #     sanitized_docs,
    #     baseEntityLabel=True,
    #     include_source=True,
    # )
    # print(f"Added {filename} to graph.")

In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en",
    model_kwargs = {"device": "cpu"}
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
def invoke_vector_retriever(query:str, k:int=10):
    vector_retriever = vector_index.as_retriever(search_kwargs={"k": k})
    return vector_retriever.invoke(query)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3380.79it/s]


ValueError: Did not find url, please add an environment variable `NEO4J_URI` which contains it, or pass `url` as a named parameter.

In [44]:
driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))

def create_fulltext_index(tx):
    query = '''
    CREATE FULLTEXT INDEX `fulltext_entity_id` 
    FOR (n:__Entity__) 
    ON EACH [n.id];
    '''
    tx.run(query)

# Function to execute the query
def create_index():
    with driver.session(database="local2") as session:
        session.execute_write(create_fulltext_index)
        print("Fulltext index created successfully.")

# Call the function to create the index
try:
    create_index()
except:
    print("Index creation failed")
    pass

# Close the driver connection
driver.close()

Fulltext index created successfully.


In [35]:
llm = ChatOllama(
    model="qwen3.6:latest",
    base_url="http://192.168.178.67:11434",
    temperature=0,
    reasoning=False,
    keep_alive="24h",
    # format="json"
)
llm.invoke("Say hello in one sentence!")

AIMessage(content='Hello!', additional_kwargs={}, response_metadata={'model': 'qwen3.6:latest', 'created_at': '2026-07-10T15:52:25.0215231Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1371776200, 'load_duration': 657650300, 'prompt_eval_count': 18, 'prompt_eval_duration': 353369000, 'eval_count': 3, 'eval_duration': 357722000, 'logprobs': None, 'model_name': 'qwen3.6:latest', 'model_provider': 'ollama'}, id='lc_run--019f4cba-e024-7cc1-9a0b-904fdaa07406-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 3, 'total_tokens': 21})

In [16]:
from langchain_anthropic import ChatAnthropic

cloud_llm = ChatAnthropic(
    model_name="claude-haiku-4-5-20251001",
    temperature=0,
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

In [17]:

class Entities(BaseModel):
    """Entity mentions relevant for graph retrieval."""
    names: list[str] = Field(
        ...,
        description="All entity names explicitly mentioned in the question that may appear as graph nodes."
    )

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You extract graph entity mentions from questions. Return all explicitly mentioned entities that may exist as graph nodes."
    ),
    (
        "human",
        "Extract entities from this question: {question}"
    ),
])


entity_chain = prompt | cloud_llm.with_structured_output(Entities, method="function_calling")

In [18]:
entity_chain.invoke("How are Sherlock Holmes and Irene Adler related?")

Entities(names=['Sherlock Holmes', 'Irene Adler'])

In [19]:
def generate_full_text_query(input: str) -> str:
    words = [el for el in remove_lucene_chars(input).split() if el]
    if not words:
        return ""
    full_text_query = " AND ".join([f"{word}~2" for word in words])
    print(f"Generated Query: {full_text_query}")
    return full_text_query.strip()


# Fulltext index query
# # def graph_retriever(question: str, k: int = 10) -> str:
#     all_outputs = []
#     entities = entity_chain.invoke(question)

#     for entity in entities.names:
#         lucene_query = generate_full_text_query(entity)
#         response = graph.query(
#             """
#             CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})
#             YIELD node, score
#             WITH node, score AS node_score
#             CALL (node) {
#               MATCH (node)-[r]->(neighbor)
#               WHERE type(r) <> 'MENTIONS'
#               RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
#               UNION ALL
#               MATCH (node)<-[r]-(neighbor)
#               WHERE type(r) <> 'MENTIONS'
#               RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
#             }
#             RETURN output, node_score
#             ORDER BY node_score DESC
#             """,
#             {"query": lucene_query},
#         )

#         all_outputs.extend((row["node_score"], row["output"]) for row in response)
#         all_outputs.sort(key=lambda x: x[0], reverse=True)

#         print(all_outputs)

#     return "\n".join(all_outputs[:k])


def graph_retriever(question: str, k: int = 10) -> str:
    result = []
    entities = entity_chain.invoke(question)
    if not entities:
        raise Exception("No entities present")

    for entity in entities.names:
        response = graph.query(
            """
        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})
        YIELD node, score
        CALL {
          WITH node
          MATCH (node)-[r:!MENTIONS]->(neighbor)
          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
          UNION
          WITH node
          MATCH (node)<-[r]-(neighbor)
          RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
        }
        RETURN output
        LIMIT 50
        """,
            {"query": generate_full_text_query(entity)},
        )

        result.extend(row["output"] for row in response)

    return "\n".join(sorted(set(result))[:k])

In [20]:
graph_retriever("Jabez Wilson")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r:!MENTIONS]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<

Generated Query: Jabez~2 AND Wilson~2


''

In [38]:
def full_retriever(question: str):
    graph_data = graph_retriever(question, k=50)
    vector_data = [el.page_content for el in invoke_vector_retriever(question, k=50)]
    final_data = f"""Graph data:
{graph_data}
vector data:
{"#Document ". join(vector_data)}
    """
    return final_data

In [39]:
template = """Answer the question based only on the following context.

Context: {context}

Question: {question}

If a direct graph relation answers the question, prefer it.
Use natural language and be concise.
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
        {
            "context": full_retriever,
            "question": RunnablePassthrough(),
        }
    | prompt
    | llm
    | StrOutputParser()
)

In [45]:
chain.invoke(input="Who is Jabez Wilson's assistant in 'The Red-Headed League'?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r:!MENTIONS]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<

Generated Query: Jabez~2 AND Wilson~2
Generated Query: The~2 AND Red~2 AND Headed~2 AND League~2


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

'Vincent Spaulding'

In [46]:
def format_docs(relevant_docs):
    # return "\n".join(doc.payload["page_content"] for doc in relevant_docs)
    return "\n".join(doc.page_content for doc in relevant_docs)


def answer_with_context(query):
    context = full_retriever(query)

    # docs =  retrieve_with_reranking(query, k=20)
    # context = format_docs(docs)

    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": query}
    )

    return {
        "query": query,
        "context": context,
        "answer": answer,
        "docs": docs,
    }

In [60]:
answer_with_context("Who is Jabez Wilson's assistant in 'The Red-Headed League'?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<-[r]-(neig

{'query': "Who is Jabez Wilson's assistant in 'The Red-Headed League'?",
 'context': "Graph data:\n04cf4e3a889b8325bb0df63a1a905a35 - MENTIONS -> Jabez Wilson\n1d8f87dbdbf4eebe856b397517c27960 - MENTIONS -> Jabez Wilson\n50d7fdb8838d871a13a1841cd78bdd6e - MENTIONS -> Mr. Jabez Wilson\n6b24a713e8e8ff01e37fe534cf510a89 - MENTIONS -> Mr. Jabez Wilson\n7eb7c5e4a56adc3d1defcff8b0056476 - MENTIONS -> Mr. Jabez Wilson\n936d4df8ee41ef7fe59bfe05f648f835 - MENTIONS -> Mr. Jabez Wilson\nJabez Wilson - EMPLOYS -> Vincent Spaulding\nJabez Wilson - LOCATED_AT -> Coburg Square\nJabez Wilson - OPERATES_BUSINESS_AT -> Saxe-Coburg Square\nLeague offices - ASSOCIATED_WITH -> Mr. Jabez Wilson\nMr. Jabez Wilson - DESTINATION -> Pope’s Court\nMr. Jabez Wilson - EMPLOYED -> Vincent Spaulding\nMr. Jabez Wilson - HASFEATURE -> fish tattoo\nMr. Jabez Wilson - MEMBEROF -> Freemasonry\nMr. Jabez Wilson - NARRATOR_OF -> Dr. Watson\nMr. Jabez Wilson - OCCUPATION -> ship's carpenter\nMr. Jabez Wilson - POSSESSES -> 

In [1]:
import json

qa_path = "../data/qa/"
# for file in os.listdir(qa_path):
with open(f"{qa_path}qa_chapter_1.json") as f:
    json_qa = json.loads(f.read())



In [2]:
json_qa

[{'id': 1,
  'question': "In the opening of 'A Scandal in Bohemia' what is Holmes's special relationship to Irene Adler, and what nickname does Watson say Holmes uses for her?",
  'answer': "Holmes regards Irene Adler as the woman who eclipses the whole of her sex in his eyes. Watson says Holmes refers to her as 'the woman'.",
  'type': 'single-hop',
  'difficulty': 'easy',
  'evidence_hint': "A Scandal in Bohemia; Irene Adler; 'the woman'"},
 {'id': 2,
  'question': 'Why does the King of Bohemia visit Sherlock Holmes in disguise?',
  'answer': "He comes incognito because the matter is delicate and could compromise one of Europe's reigning families. He wants Holmes's help recovering compromising letters and a photograph tied to Irene Adler.",
  'type': 'single-hop',
  'difficulty': 'easy',
  'evidence_hint': 'King of Bohemia; incognito; compromising letters; photograph'},
 {'id': 3,
  'question': 'What clue in the anonymous note leads Holmes to conclude that the paper was made in Bohem

### Evaluate vector retrieval

In [ ]:
vector_qa_data = []
for pair in json_qa:
    context = [el.page_content for el in invoke_vector_retriever(pair["question"], k=50)]
    answer = (prompt | cloud_llm | StrOutputParser()).invoke(
        {"context": context, "question": pair["question"]}
    )
    vector_qa_data.append(
        {
            "user_input": pair["question"],
            # "retrieved_contexts": [res.payload["page_content"] for res in response["docs"]],
            "retrieved_contexts": context,
            "response": answer,
            "reference": pair["answer"],
        }
    )
    print(f"processed {len(vector_qa_data)}/{len(json_qa[:10])} qa pairs")
vector_evaluation_dataset = EvaluationDataset.from_list(vector_qa_data)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

processed 1/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

processed 2/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

...

processed 7/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

processed 8/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

processed 9/9 qa pairs


### Evaluate graph retrieval

In [21]:
graph_qa_data = []
for pair in json_qa:
    graph_text = graph_retriever(pair["question"])
    print("retrieved graph information...")
    context = [line for line in graph_text.split("\n") if line.strip()]
    answer = (prompt | cloud_llm | StrOutputParser()).invoke(
        {"context": context, "question": pair["question"]}
    )
    print("generated answer...")
    graph_qa_data.append(
        {
            "user_input": pair["question"],
            # "retrieved_contexts": [res.payload["page_content"] for res in response["docs"]],
            "retrieved_contexts": context,
            "response": answer,
            "reference": pair["answer"],
        }
    )
    print(f"processed {len(graph_qa_data)}/{len(json_qa[:10])} qa pairs")
graph_evaluation_dataset = EvaluationDataset.from_list(graph_qa_data)

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r:!MENTIONS]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<

Generated Query: A~2 AND Scandal~2 AND in~2 AND Bohemia~2
Generated Query: Holmes~2
Generated Query: Irene~2 AND Adler~2
Generated Query: Watson~2
retrieved graph information...
generated answer...
processed 1/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r:!MENTIONS]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<

Generated Query: King~2 AND of~2 AND Bohemia~2
Generated Query: Sherlock~2 AND Holmes~2
retrieved graph information...
generated answer...
processed 2/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r:!MENTIONS]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<

...

Generated Query: Holmes~2
Generated Query: Watson~2
Generated Query: Briony~2 AND Lodge~2
Generated Query: smoke~2 AND rocket~2
retrieved graph information...
generated answer...
processed 7/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r:!MENTIONS]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<

Generated Query: Irene~2 AND Adler~2
Generated Query: King~2 AND of~2 AND Bohemia~2
retrieved graph information...
generated answer...
processed 8/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=4, column=9, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})\n        YIELD node, score\n        CALL {\n          WITH node\n          MATCH (node)-[r:!MENTIONS]->(neighbor)\n          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n          UNION\n          WITH node\n          MATCH (node)<

Generated Query: Holmes~2
Generated Query: Irene~2 AND Adler~2
retrieved graph information...
generated answer...
processed 9/9 qa pairs


### Evaluate hybrid retrieval 

In [28]:
hybrid_qa_data = []
for pair in json_qa:
    vector_context = [el.page_content for el in invoke_vector_retriever(pair["question"], k=50)]
    graph_text = graph_retriever(pair["question"])
    graph_context = [line for line in graph_text.split("\n") if line.strip()]
    hybrid_context = graph_context + vector_context

    answer = (prompt | cloud_llm | StrOutputParser()).invoke(
        {"context": hybrid_context, "question": pair["question"]}
    )

    hybrid_qa_data.append(
        {
            "user_input": pair["question"],
            # "retrieved_contexts": [res.payload["page_content"] for res in response["docs"]],
            "retrieved_contexts": hybrid_context,
            "response": answer,
            "reference": pair["answer"],
        }
    )
    print(f"processed {len(hybrid_qa_data)}/{len(json_qa[:10])} qa pairs")
hybrid_evaluation_dataset = EvaluationDataset.from_list(hybrid_qa_data)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Generated Query: A~2 AND Scandal~2 AND in~2 AND Bohemia~2
Generated Query: Holmes~2
Generated Query: Irene~2 AND Adler~2
Generated Query: Watson~2
processed 1/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Generated Query: King~2 AND of~2 AND Bohemia~2
Generated Query: Sherlock~2 AND Holmes~2
processed 2/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

...

Generated Query: Holmes~2
Generated Query: Watson~2
Generated Query: Briony~2 AND Lodge~2
Generated Query: smoke~2 AND rocket~2
processed 7/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Generated Query: Irene~2 AND Adler~2
Generated Query: King~2 AND of~2 AND Bohemia~2
processed 8/9 qa pairs


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Generated Query: Holmes~2
Generated Query: Irene~2 AND Adler~2
processed 9/9 qa pairs


In [29]:
import pandas
from ragas import evaluate, RunConfig
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
)
from ragas.metrics import AspectCritic
from langchain_anthropic import ChatAnthropic

evaluation_llm = ChatAnthropic(
    model_name="claude-haiku-4-5-20251001",
    temperature=0,
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)
judge_llm = LangchainLLMWrapper(evaluation_llm)
judge_embedding = LangchainEmbeddingsWrapper(embeddings)

scores = evaluate(
    hybrid_evaluation_dataset,
    metrics=[
        answer_relevancy,
        faithfulness,
        context_precision,
        context_recall,
    ],
    llm=judge_llm,
    embeddings=judge_embedding,
)

# print(rows)
print(scores)

/tmp/ipykernel_49693/1794412133.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_49693/1794412133.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_49693/1794412133.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_49693/1794412133.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be rem

{'answer_relevancy': 0.7572, 'faithfulness': 0.3815, 'context_precision': 0.0000, 'context_recall': 0.0000}


In [59]:
import pandas
from ragas import evaluate, RunConfig
from ragas.metrics import _noise_sensitivity, _context_entities_recall
from ragas.metrics import AspectCritic
from langchain_anthropic import ChatAnthropic

evaluation_llm = ChatAnthropic(
    model_name="claude-haiku-4-5-20251001",
    temperature=0,
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)
judge_llm = LangchainLLMWrapper(evaluation_llm)
judge_embedding = LangchainEmbeddingsWrapper(embeddings)


graph_groundedness = AspectCritic(
    name="graph_groundedness",
    definition=(
        "Does the answer rely only on facts that are explicitly supported "
        "by the retrieved graph relations and retrieved text context?"
    ),
    llm=evaluation_llm,
)

scores = evaluate(
    vector_evaluation_dataset,
    metrics=[
        _noise_sensitivity.NoiseSensitivity(),
        _context_entities_recall.ContextEntityRecall(),
        graph_groundedness,
    ],
    llm=judge_llm,
    embeddings=judge_embedding,
)

# print(rows)
print(scores)

/tmp/ipykernel_38340/102916445.py:4: DeprecationWarning: Importing AspectCritic from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AspectCritic
  from ragas.metrics import AspectCritic
/tmp/ipykernel_38340/102916445.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(evaluation_llm)
/tmp/ipykernel_38340/102916445.py:13: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embedding

{'noise_sensitivity(mode=relevant)': nan, 'context_entity_recall': 0.1250, 'graph_groundedness': nan}


In [ ]:
from dataclasses import dataclass, field
import typing as t
import re

from ragas.metrics.base import MetricType, SingleTurnMetric, Metric
from ragas.dataset_schema import SingleTurnSample

@dataclass
class EntityCoverageMetric(Metric, SingleTurnMetric):
    name: str = "entity_coverage"
    _required_columns: t.Dict[MetricType, t.Set[str]] = field(
        default_factory=lambda: {
            MetricType.SINGLE_TURN: {"retrieved_contexts", "reference"}
        }
    )

    async def _single_turn_ascore(self, sample: SingleTurnSample, callbacks=None) -> float:
        gold_entities = set(re.findall(r"\[(.*?)\]", sample.reference))
        if not gold_entities:
            return 0.0

        context_text = " ".join(sample.retrieved_contexts).lower()
        hits = sum(1 for ent in gold_entities if ent.lower() in context_text)
        return hits / len(gold_entities)

### Graph quality

In [9]:
from collections import Counter
from typing import Tuple
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
import json

def node_key(node: Node) -> Tuple[str, str]:
    """
    Simple node identity: (id, type).
    Extend here if you want to include specific properties.
    """
    return (str(node.id).strip(), str(node.type).strip())

def rel_key(rel: Relationship) -> Tuple[str, str, str, str, str]:
    """
    Simple relationship identity: (source.id, source.type, type, target.id, target.type).
    """
    return (
        str(rel.source.id).strip(),
        str(rel.source.type).strip(),
        str(rel.type).strip(),
        str(rel.target.id).strip(),
        str(rel.target.type).strip(),
    )


def precision_recall_f1(tp: int, fp: int, fn: int) -> dict:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1}


def evaluate_nodes(gold_graph: GraphDocument, pred_graph: GraphDocument) -> dict:
    gold_nodes = {node_key(n) for n in gold_graph.nodes}
    pred_nodes = {node_key(n) for n in pred_graph.nodes}

    tp = len(gold_nodes & pred_nodes)
    fp = len(pred_nodes - gold_nodes)
    fn = len(gold_nodes - pred_nodes)

    scores = precision_recall_f1(tp, fp, fn)
    scores.update({
        "gold_node_count": len(gold_nodes),
        "pred_node_count": len(pred_nodes),
        "tp_nodes": tp,
        "fp_nodes": fp,
        "fn_nodes": fn,
    })
    return scores


def evaluate_relationships(gold_graph: GraphDocument, pred_graph: GraphDocument) -> dict:
    gold_rels = {rel_key(r) for r in gold_graph.relationships}
    pred_rels = {rel_key(r) for r in pred_graph.relationships}

    tp = len(gold_rels & pred_rels)
    fp = len(pred_rels - gold_rels)
    fn = len(gold_rels - pred_rels)

    scores = precision_recall_f1(tp, fp, fn)
    scores.update({
        "gold_rel_count": len(gold_rels),
        "pred_rel_count": len(pred_rels),
        "tp_rels": tp,
        "fp_rels": fp,
        "fn_rels": fn,
    })
    return scores

In [ ]:
def evaluate_graph_quality(gold_graph: GraphDocument, pred_graph: GraphDocument) -> dict:
    node_scores = evaluate_nodes(gold_graph, pred_graph)
    rel_scores = evaluate_relationships(gold_graph, pred_graph)
    return {
        "nodes": node_scores,
        "relationships": rel_scores,
    }

In [83]:
def merge_graph_documents(graph_docs: list[GraphDocument]) -> GraphDocument:
    resolved_graph_docs = []
    for gd in graph_docs:
        resolved_graph_docs.append(resolve_results(gd))

    all_nodes = []
    all_rels = []
    # You can pick any source; here we just keep the first document's source
    source = graph_docs[0].source if graph_docs else None

    for gd in graph_docs:
        all_nodes.extend(gd.nodes)
        all_rels.extend(gd.relationships)

    return GraphDocument(nodes=all_nodes, relationships=all_rels, source=source)

def get_nodes_ids(graph_docs:any):
    # resolve graph docs
    resolved_graph_docs = []
    for gd in graph_docs:
        resolved_graph_docs.append(resolve_results(gd))
    # merge graph docs
    merged_graph_docs = merge_graph_documents(resolved_graph_docs)

    # return only node ids
    all_nodes_names = []
    for node in merged_graph_docs.nodes:
        if node.id not in all_nodes_names:
            all_nodes_names.append(node.id)
    return all_nodes_names

def get_rel_types(graph_docs:any):
    all_rel_types = []
    for rel in merge_graph_documents(graph_docs).relationships:
        if rel.type not in all_rel_types:
            all_rel_types.append(rel.type)
    return all_rel_types

extracted_docs = sanitized_docs

with open("../data/gold_standard/chapter_1.txt", "r", encoding="utf-8") as file_in:
    gold_standard_text = file_in.read()

gold_standard_docs = convert_to_graph_doc(gold_standard_text)

pred_graph_ch1 = merge_graph_documents(extracted_docs[:15])
extracted_ids = get_nodes_ids(sanitized_docs[:15])
extracted_types = get_rel_types(sanitized_docs[:15])
print(extracted_types)


gold_graph_ch1 = merge_graph_documents(gold_standard_docs[:15])
gold = get_nodes_ids(gold_standard_docs[:15])
gold_types = get_rel_types(gold_standard_docs[:15])
print(gold_types)
# print(f"extracted: {len(extracted)}, gold: {len(gold)}")

scores_ch1 = evaluate_graph_quality(gold_graph_ch1, pred_graph_ch1)
print(json.dumps(scores_ch1, indent=2, ensure_ascii=False))

['IS ASSOCIATED WITH', 'RESIDES IN', 'INVOLVED IN', 'ASSOCIATED WITH', 'WORKED FOR', 'VISITED', 'PASSED THROUGH', 'OCCURRED ON', 'REMARKED TO', 'OBSERVED_WEIGHT_GAIN', 'DEDUCED_ACTIVITY', 'IDENTIFIED_AS_SERVANT', 'EMPLOYED_BY', 'VISITED_LOCATION', 'DEDUCES FACTS ABOUT', 'OBSERVES REASONING OF', 'SPEAKS_TO', 'SITS_IN', 'LIGHTS', 'THROWS', 'WAS_ON', 'LEADS_TO', 'LOCATED_IN', 'OBSERVED', 'WILL_CALL_UPON', 'SCHEDULED_AT', 'SCHEDULED_ON', 'PROVIDED_SERVICES_TO', 'EXAMINED', 'DEDUCES_FROM', 'ASKS_ABOUT', 'CONSULTED', 'LOCATED IN', 'DIED IN', 'IDENTIFIED ORIGIN OF PAPER', 'IDENTIFIED NATIONALITY OF WRITER', 'CALLS_BY_ALIAS', 'ADDRESSES_BY_ROLE', 'ADDRESSED_BY', 'WORE_AS_PART_OF', 'CARRIED', 'WORE', 'RECEIVED', 'COLLEAGUE', 'INFLUENCE', 'MEMBER_OF', 'RULER_OF', 'REIGNING_FAMILY_OF', 'ORIGIN', 'CONSULTING', 'ACQUAINTANCE', 'ADDRESSING', 'BIOGRAPHY_SOURCE', 'INDEX_NEIGHBOR', 'BIRTH_PLACE', 'AFFILIATION', 'EMPLOYMENT', 'RESIDENCE', 'WROTE_LETTERS_TO', 'POSSESSES', 'IS IN', 'OWNS', 'IS ENGAGED TO'

In [62]:
ext_graph = Neo4jGraph(database="local-3-ext")
ext_graph.add_graph_documents(
    sanitized_docs[:15],
    baseEntityLabel=True,
    include_source=True,
)

In [65]:
gold_graph = Neo4jGraph(database="local-3-gold")
gold_graph.add_graph_documents(
    gold_standard_docs[:15],
    baseEntityLabel=True,
    include_source=True,
)